# JetBot - Data collection without gamecontroller

In this notebook we'll collect training data for CNN VAE. The training data save to dataset directory.

## Import module



In [ ]:
import os
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Robot, Camera, bgr8_to_jpeg
import time
import cv2
import numpy as np
import glob
from uuid import uuid1

## Initialize Camera (Shared for Calibration and Data Collection)

Initialize the camera here for use in both calibration and data collection. Image size is 1280 x 960 for higher quality.

In [ ]:
camera = Camera.instance(width=1280, height=960)
image = widgets.Image(format='jpeg', width=320, height=240)  # Display smaller for UI
camera_link = traitlets.dlink((camera,'value'), (image,'value'), transform=bgr8_to_jpeg)
display(image)  # Display camera feed for monitoring

## Camera Calibration Setup

Before collecting data, calibrate the camera for undistortion. Print a checkerboard pattern (e.g., 7x10 squares, 25mm size) and capture 20-50 images from different angles/distances.

Use the toggle button below to start/stop logging calibration images.

In [ ]:
calib_log_button = widgets.ToggleButton(value=False, description='enable calib logging')
display(calib_log_button)

CALIB_DIR = 'calib_images'
try:
    os.makedirs(CALIB_DIR)
except FileExistsError:
    print('Calibration directory already exists')

calib_count_box = widgets.IntText(value=len(os.listdir(CALIB_DIR)))
calib_count_label = widgets.Label(value='Number calib images:')
calib_count_panel = widgets.HBox([calib_count_label, calib_count_box])
display(calib_count_panel)

def save_calib_record(change):
    if calib_log_button.value:
        image_name = '{}.jpg'.format(uuid1())
        image_path = os.path.join(CALIB_DIR, image_name)
        save_image = bgr8_to_jpeg(change['new'])
        with open(image_path, 'wb') as f:
            f.write(save_image)
        calib_count_box.value = len(os.listdir(CALIB_DIR))

camera.observe(save_calib_record, names='value')

## Stop Calibration Capture

Run this to stop logging calibration images.

In [ ]:
camera.unobserve(save_calib_record, names='value')

## Perform Calibration

Run this cell to calibrate using the captured images. Adjust CHECKERBOARD if your pattern differs.

Copy the printed K and D values into the data collection cell below (replace the example values).

In [ ]:
# Checkerboard size (inner corners)
CHECKERBOARD = (7, 10)  # Adjust to your pattern

# Prepare object points (3D)
objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2) * 25.0  # Square size in mm

objpoints = []  # 3D points
imgpoints = []  # 2D points

# Load images
images = glob.glob(f'{CALIB_DIR}/*.jpg')

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)
    if ret:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)

# Calibrate
if len(objpoints) > 0:
    ret, K, D, rvecs, tvecs = cv2.fisheye.calibrate(objpoints, imgpoints, gray.shape[::-1], None, None)
    print("Intrinsic Matrix K:\n", K)
    print("Distortion D:\n", D)
else:
    print("No valid checkerboard images found. Capture more images.")

## Show log_button

If you enable log_button then start recording images.


In [ ]:
log_button = widgets.ToggleButton(value=False, description='enable logging')
display(log_button)

## UI Widget for Data Collection


In [ ]:
DATASET_DIR = 'dataset'
try:
    os.makedirs(DATASET_DIR)
except FileExistsError:
    print('Dataset directory already exists')

dataset = DATASET_DIR
layout = widgets.Layout(width='100px', height='64px')
count_box = widgets.IntText(layout=layout, value=len(os.listdir(dataset)))
count_label = widgets.Label(layout=layout, value='Number image:')
count_panel = widgets.HBox([count_label, count_box])

panel = widgets.VBox([count_panel])
display(widgets.HBox([panel]))  # Camera feed already displayed above

## Set callback for collect the training data.

```save_record``` is callback for training data. The method set to camera observer. This callback saving the image to DATASET_DIR. When click ```enable logging``` button, this method recording training data. You can check number of training data with ```Number image text box```.

Added: Frame skipping (every 5th frame), fisheye undistortion (using K and D from calibration—paste values here), and basic augmentation (brightness/contrast).

In [ ]:
# Paste your calibrated values here from the calibration output above
fx, fy, cx, cy = 1046.0, 972.0, 640.0, 473.0  # Example defaults; replace with your K values
k1, k2, p1, p2 = -0.02, 0.05, 0.0, 0.0  # Example defaults; replace with your D values

K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
D = np.array([k1, k2, p1, p2], dtype=np.float64)

frame_counter = 0
skip_frames = 5  # Save every 5th frame to reduce redundancy

def save_record(change):                
    global frame_counter
    if log_button.value:
        frame_counter += 1
        if frame_counter % skip_frames != 0:
            return
        
        image = change['new']
        
        # Undistort fisheye
        undistorted = cv2.fisheye.undistortImage(image, K, D, None, K)
        
        # Simple augmentation: random brightness/contrast
        alpha = 1.0 + np.random.uniform(-0.3, 0.3)  # Contrast
        beta = np.random.uniform(-50, 50)  # Brightness
        augmented = cv2.convertScaleAbs(undistorted, alpha=alpha, beta=beta)
        
        image_name = '{}.jpg'.format(uuid1())
        image_path = os.path.join(DATASET_DIR, image_name)
        save_image = bgr8_to_jpeg(augmented)
        with open(image_path, 'wb') as f:
            f.write(save_image)
        count_box.value = len(os.listdir(dataset)) 


save_record({'new': camera.value})
camera.observe(save_record, names='value')

## Cleanup

After collecting enough data. cleanup camera observer and stop all motor.

In [ ]:
camera.unobserve(save_record, names='value')
camera_link.unlink()
camera.stop()

## Create dataset.zip file 

In [ ]:
import datetime
def timestr():
    return str(datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S'))

!zip -r -q jetbot_{DATASET_DIR}_{timestr()}.zip {DATASET_DIR}